In [19]:
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

In [20]:
class State(TypedDict):
    item: str
    price: float
    qty: int
    available: bool
    total_amount: float

In [21]:
shopping_list = [
    "T-Shirts",
    "Jeans",
    "Hoodie",
    "Formal Shirt",
    "Casual Trousers",
    "Jacket",
    "Socks",
    "Sneakers",
    "A4 Notebooks",
    "Gel Pens (Blue/Black)",
    "Pencil Pouch",
    "Highlighters",
    "Sticky Notes",
    "Paper Clips",
    "Stapler",
    "Whiteboard Markers",
]

In [22]:
def check_stock(state: State):
    return {
        "available": state['item'] in shopping_list
    }

In [23]:
def should_calculate(state: State):
    if state["available"]:
        return "generate_bill"
    return END

In [30]:
def generate_bill(state: State):
    amt = state['qty'] * state["price"]
    return {"total_amount": amt}

In [31]:
builder = StateGraph(State)

builder.add_node("check_stock", check_stock)
builder.add_node("generate_bill", generate_bill)

builder.add_edge(START, "check_stock")
builder.add_conditional_edges("check_stock",              should_calculate, {
        "generate_bill": "generate_bill",
        END: END,
    },
    )
builder.add_edge("generate_bill", END)

In [32]:
graph = builder.compile()

In [33]:
res1 = graph.invoke(
    {
        "item": "Jeans",
        "price": 1200,
        "qty": 2,
    }
)

In [34]:
print(res1)

{'item': 'Jeans', 'price': 1200, 'qty': 2, 'available': True, 'total_amount': 2400}


In [35]:
for event in graph.stream(
    {
        "item": "Jeans",
        "price": 1200,
        "qty": 2,
    },
    stream_mode="updates",
):
    print(event)

{'check_stock': {'available': True}}
{'generate_bill': {'total_amount': 2400}}
